In [3]:
import os
MONGO_URI = os.environ["MONGO_URI"]


In [ ]:
import os
CLAUDE_API_KEY = os.environ["ANTHROPIC_API_KEY"]


In [ ]:
import os
# =====================================================
# IMPORTS
# =====================================================

from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import anthropic
import numpy as np

# =====================================================
# CONFIG
# =====================================================

MONGO_URI = os.environ["MONGO_URI"]

DB_NAME = "365"

COLLECTION_NAME = "embedding6"

# SAME MODEL AS EMBEDDING SCRIPT
EMBED_MODEL = "intfloat/multilingual-e5-base"

CLAUDE_API_KEY = os.environ["ANTHROPIC_API_KEY"]
CLAUDE_MODEL = "claude-haiku-4-5"

TOP_K = 5

# nếu similarity thấp hơn threshold
# thì Claude sẽ trả lời theo hiểu biết chung
SIMILARITY_THRESHOLD = 0.72

# =====================================================
# CONNECT MONGODB
# =====================================================

print("Connecting MongoDB...")

client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=30000
)

db = client[DB_NAME]

collection = db[COLLECTION_NAME]

print("MongoDB connected.")

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

print("Loading embedding model...")

embedder = SentenceTransformer(
    EMBED_MODEL
)

print("Embedding model loaded.")

# =====================================================
# LOAD CLAUDE
# =====================================================

print("Loading Claude client...")

claude = anthropic.Anthropic(
    api_key=CLAUDE_API_KEY
)

print("Claude client loaded.")

# =====================================================
# USER QUERY
# =====================================================

query = input("\nQuestion: ")

# =====================================================
# CREATE QUERY EMBEDDING
# =====================================================

print("\nCreating query embedding...")

query_embedding = embedder.encode(

    f"query: {query}",

    normalize_embeddings=True
)

# =====================================================
# LOAD ALL EMBEDDINGS
# =====================================================

print("\nLoading embeddings from MongoDB...")

documents = list(

    collection.find(

        {},

        {
            "_id": 0,

            "hotel_name": 1,

            "chunk_type": 1,

            "text": 1,

            "data": 1,

            "embedding": 1
        }
    )
)

print(f"Loaded {len(documents)} chunks.")

# =====================================================
# COSINE SIMILARITY
# =====================================================

results = []

for doc in documents:

    embedding = doc.get(
        "embedding",
        []
    )

    if not embedding:
        continue

    similarity = cosine_similarity(

        [query_embedding],

        [embedding]

    )[0][0]

    results.append({

        "hotel_name":
            doc.get(
                "hotel_name",
                ""
            ),

        "chunk_type":
            doc.get(
                "chunk_type",
                ""
            ),

        "text":
            doc.get(
                "text",
                ""
            ),

        "score":
            float(similarity)
    })

# =====================================================
# SORT RESULTS
# =====================================================

results = sorted(

    results,

    key=lambda x: x["score"],

    reverse=True
)

top_results = results[:TOP_K]

# =====================================================
# DEBUG RETRIEVED CHUNKS
# =====================================================

print("\n" + "=" * 100)
print("TOP RETRIEVED CHUNKS")
print("=" * 100)

for idx, r in enumerate(
    top_results,
    start=1
):

    print("\n" + "-" * 100)

    print(f"RANK        : {idx}")

    print(
        f"HOTEL       : "
        f"{r['hotel_name']}"
    )

    print(
        f"CHUNK TYPE  : "
        f"{r['chunk_type']}"
    )

    print(
        f"SIMILARITY  : "
        f"{round(r['score'], 4)}"
    )

    print("\nTEXT:\n")

    print(r["text"])

# =====================================================
# BEST SCORE
# =====================================================

best_score = 0

if top_results:

    best_score = top_results[0]["score"]

print("\nBest similarity:", round(best_score, 4))

# =====================================================
# BUILD CONTEXT
# =====================================================

context = ""

for r in top_results:

    context += f"""

====================================================
HOTEL
====================================================

HOTEL NAME:
{r["hotel_name"]}

CHUNK TYPE:
{r["chunk_type"]}

SIMILARITY:
{round(r["score"], 4)}

CONTENT:
{r["text"]}

"""

# =====================================================
# CASE 1
# GOOD MATCH -> USE RAG
# =====================================================

if best_score >= SIMILARITY_THRESHOLD:

    print("\nUsing RAG context...")
    prompt = f"""
    You are a premium AI travel assistant.

    Write responses in a beautiful, friendly,
    well-structured format.

    STYLE:
    
    - Use markdown headings
    - Use bullet points
    - Use emojis naturally
    - Make the answer visually pleasant
    - Do not sound robotic
    - Expand useful details
    - Highlight recommendations
    - Keep spacing clean

    IMPORTANT:

    - Use ONLY information from provided context
    - Never hallucinate
    - If information is missing, say so naturally

    ====================================================
    CONTEXT
    ====================================================

    {context}

    ====================================================
    QUESTION
    ====================================================

    {query}

    ====================================================
    ANSWER
    ====================================================
    """

# =====================================================
# CASE 2
# LOW MATCH -> NORMAL CLAUDE
# =====================================================

else:

    print("\nNo strong chunk match found.")
    print("Using Claude general knowledge...")

    prompt = f"""
You are a helpful hotel assistant.

The vector database did not find
reliable hotel contract information.

Answer the user's question normally
using your own understanding.

QUESTION:
{query}
"""

# =====================================================
# CLAUDE GENERATION
# =====================================================

print("\nGenerating answer...")

response = claude.messages.create(

    model=CLAUDE_MODEL,

    max_tokens=1000,

    temperature=0,

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.content[0].text

# =====================================================
# FINAL ANSWER
# =====================================================

print("\n" + "=" * 100)
print("FINAL ANSWER")
print("=" * 100)

print(answer)

# continuous chat

In [ ]:
import os
# =====================================================
# IMPORTS & CONFIG (Giữ nguyên như cũ)
# =====================================================
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import anthropic
import numpy as np

MONGO_URI = os.environ["MONGO_URI"]
DB_NAME = "365"
COLLECTION_NAME = "embedding6"
EMBED_MODEL = "intfloat/multilingual-e5-base"
CLAUDE_API_KEY = os.environ["ANTHROPIC_API_KEY"]
CLAUDE_MODEL = "claude-haiku-4-5"
TOP_K = 5
SIMILARITY_THRESHOLD = 0.72

# =====================================================
# SYSTEM PROMPT (Tách riêng ra để đưa vào tham số system)
# =====================================================
SYSTEM_INSTRUCTIONS = """Bạn là một chuyên gia du lịch AI sành điệu, hiện đại và thân thiện.
Nhiệm vụ của bạn là trình bày thông tin khách sạn/du lịch thật đẹp mắt, dễ đọc và mang phong cách "local".

QUY TẮC ĐỊNH DẠNG (BẮT BUỘC):
1. Mở bài: Viết 1-2 câu giới thiệu ngắn gọn. Sử dụng ký hiệu ≈ trước tên địa điểm (VD: ≈ Khách sạn A là...).
2. Cấu trúc bài viết: Chia thành các mục rõ ràng với tiêu đề in đậm kèm Emoji phù hợp (VD: **Tổng quan nhanh** 🌿, **Lưu ý nhỏ** ✅).
3. Danh sách (Bullet points): Luôn dùng bullet points, bôi đậm từ khóa chính. Thêm emoji nhỏ ở đầu dòng nếu liệt kê chi tiết.
4. Giọng văn: Trẻ trung, tự nhiên, gần gũi (dùng "vibe", "rất đã", "đúng chất") nhưng lịch sự.
5. Kết bài: Đặt MỘT câu hỏi gợi mở nhẹ nhàng kèm bullet ở cuối để hướng dẫn người dùng bước tiếp theo.
6. TUYỆT ĐỐI KHÔNG bịa đặt thông tin. Nếu không có dữ liệu, hãy trả lời khéo léo dựa trên kiến thức chung nhưng phải giữ đúng format.
"""

# =====================================================
# INITIALIZE SERVICES
# =====================================================
print("Connecting MongoDB...")
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30000)
collection = client[DB_NAME][COLLECTION_NAME]

print("Loading models & Claude client...")
embedder = SentenceTransformer(EMBED_MODEL)
claude = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
print("System Ready!\n" + "="*50)

# =====================================================
# CHAT HISTORY INITIALIZATION
# =====================================================
chat_history = [] # Mảng này sẽ lưu lại toàn bộ context cuộc hội thoại

# =====================================================
# CHAT LOOP
# =====================================================
while True:
    query = input("\nBạn (gõ 'quit' hoặc 'exit' để thoát): ")
    if query.lower() in ['quit', 'exit']:
        print("Đã thoát cuộc trò chuyện.")
        break
        
    if not query.strip():
        continue

    # 1. Embedding & Vector Search
    query_embedding = embedder.encode(f"query: {query}", normalize_embeddings=True)
    documents = list(collection.find({}, {"_id": 0, "hotel_name": 1, "chunk_type": 1, "text": 1, "embedding": 1}))
    
    results = []
    for doc in documents:
        embedding = doc.get("embedding", [])
        if not embedding: continue
        similarity = cosine_similarity([query_embedding], [embedding])[0][0]
        results.append({
            "hotel_name": doc.get("hotel_name", ""),
            "chunk_type": doc.get("chunk_type", ""),
            "text": doc.get("text", ""),
            "score": float(similarity)
        })

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    top_results = results[:TOP_K]
    best_score = top_results[0]["score"] if top_results else 0

    # 2. Chuẩn bị Context cho lượt chat hiện tại
    if best_score >= SIMILARITY_THRESHOLD:
        context_str = "\n".join([f"- HOTEL: {r['hotel_name']} | CONTENT: {r['text']}" for r in top_results])
        current_turn_prompt = f"DỮ LIỆU CUNG CẤP (CONTEXT):\n{context_str}\n\nCÂU HỎI CỦA NGƯỜI DÙNG:\n{query}"
    else:
        current_turn_prompt = query # Không có context tốt, chỉ gửi câu hỏi để Claude dùng kiến thức chung

    # 3. Nối tin nhắn mới vào lịch sử trò chuyện
    # Claude API yêu cầu list messages gồm các dict có 'role' và 'content'
    messages_for_api = chat_history + [{"role": "user", "content": current_turn_prompt}]

    # 4. Gọi Claude API
    try:
        response = claude.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=1024,
            temperature=0.3,
            system=SYSTEM_INSTRUCTIONS, # Truyền System Prompt vào đây
            messages=messages_for_api   # Truyền toàn bộ lịch sử + câu hỏi mới
        )
        
        answer = response.content[0].text
        
        # 5. In kết quả
        print("\nMindtrip AI:\n")
        print(answer)
        print("\n" + "-" * 50)
        
        # 6. CẬP NHẬT LỊCH SỬ CHO LƯỢT TIẾP THEO
        # Cần append cả câu hỏi vừa rồi và câu trả lời của AI vào mảng chat_history
        chat_history.append({"role": "user", "content": current_turn_prompt})
        chat_history.append({"role": "assistant", "content": answer})
        
    except Exception as e:
        print(f"\n[Lỗi API]: {e}")

# Ollama

In [ ]:
import os
# =====================================================
# IMPORTS
# =====================================================
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ollama  # Thay thế anthropic bằng ollama
import numpy as np

# =====================================================
# CONFIG
# =====================================================
MONGO_URI = os.environ["MONGO_URI"]
DB_NAME = "365"
COLLECTION_NAME = "embedding6"
EMBED_MODEL = "intfloat/multilingual-e5-base"

# Cấu hình Ollama
OLLAMA_MODEL = "qwen2.5:7b" # Thay bằng "qwen2", "mistral" hoặc model bạn đã tải trong Ollama
TOP_K = 5
SIMILARITY_THRESHOLD = 0.72

# =====================================================
# SYSTEM PROMPT
# =====================================================
SYSTEM_INSTRUCTIONS = """Bạn là một chuyên gia du lịch AI sành điệu, hiện đại và thân thiện.
Nhiệm vụ của bạn là trình bày thông tin khách sạn/du lịch thật đẹp mắt, dễ đọc và mang phong cách "local".

QUY TẮC ĐỊNH DẠNG (BẮT BUỘC):
1. Mở bài: Viết 1-2 câu giới thiệu ngắn gọn. Sử dụng ký hiệu ≈ trước tên địa điểm (VD: ≈ Khách sạn A là...).
2. Cấu trúc bài viết: Chia thành các mục rõ ràng với tiêu đề in đậm kèm Emoji phù hợp (VD: **Tổng quan nhanh** 🌿, **Lưu ý nhỏ** ✅).
3. Danh sách (Bullet points): Luôn dùng bullet points, bôi đậm từ khóa chính. Thêm emoji nhỏ ở đầu dòng nếu liệt kê chi tiết.
4. Giọng văn: Trẻ trung, tự nhiên, gần gũi (dùng "vibe", "rất đã", "đúng chất") nhưng lịch sự.
5. Kết bài: Đặt MỘT câu hỏi gợi mở nhẹ nhàng kèm bullet ở cuối để hướng dẫn người dùng bước tiếp theo.
6. TUYỆT ĐỐI KHÔNG bịa đặt thông tin. Nếu không có dữ liệu, hãy trả lời khéo léo dựa trên kiến thức chung nhưng phải giữ đúng format.
"""

# =====================================================
# INITIALIZE SERVICES
# =====================================================
print("Connecting MongoDB...")
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30000)
collection = client[DB_NAME][COLLECTION_NAME]

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

# Lưu ý: Không cần khởi tạo client Ollama vì thư viện gọi trực tiếp đến localhost:11434
print(f"System Ready! Đang sử dụng local model: {OLLAMA_MODEL}\n" + "="*50)

# =====================================================
# CHAT HISTORY INITIALIZATION
# =====================================================
chat_history = [] 

# =====================================================
# CHAT LOOP
# =====================================================
while True:
    query = input("\nBạn (gõ 'quit' hoặc 'exit' để thoát): ")
    if query.lower() in ['quit', 'exit']:
        print("Đã thoát cuộc trò chuyện.")
        break
        
    if not query.strip():
        continue

    # 1. Embedding & Vector Search
    query_embedding = embedder.encode(f"query: {query}", normalize_embeddings=True)
    documents = list(collection.find({}, {"_id": 0, "hotel_name": 1, "chunk_type": 1, "text": 1, "embedding": 1}))
    
    results = []
    for doc in documents:
        embedding = doc.get("embedding", [])
        if not embedding: continue
        similarity = cosine_similarity([query_embedding], [embedding])[0][0]
        results.append({
            "hotel_name": doc.get("hotel_name", ""),
            "chunk_type": doc.get("chunk_type", ""),
            "text": doc.get("text", ""),
            "score": float(similarity)
        })

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    top_results = results[:TOP_K]
    best_score = top_results[0]["score"] if top_results else 0

    # 2. Chuẩn bị Context cho lượt chat hiện tại
    if best_score >= SIMILARITY_THRESHOLD:
        context_str = "\n".join([f"- HOTEL: {r['hotel_name']} | CONTENT: {r['text']}" for r in top_results])
        current_turn_prompt = f"DỮ LIỆU CUNG CẤP (CONTEXT):\n{context_str}\n\nCÂU HỎI CỦA NGƯỜI DÙNG:\n{query}"
    else:
        current_turn_prompt = query

    # 3. Nối tin nhắn mới vào lịch sử trò chuyện
    # Với Ollama, ta đưa System Prompt vào đầu danh sách messages
    messages_for_api = [{"role": "system", "content": SYSTEM_INSTRUCTIONS}] + chat_history + [{"role": "user", "content": current_turn_prompt}]

    # 4. Gọi Ollama API
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=messages_for_api,
            options={
                "temperature": 0.3,
                "num_predict": 1024 # Tương đương max_tokens của Claude
            }
        )
        
        # Lấy text trả về từ response của Ollama
        answer = response['message']['content']
        
        # 5. In kết quả
        print("\nMindtrip AI (Local):\n")
        print(answer)
        print("\n" + "-" * 50)
        
        # 6. CẬP NHẬT LỊCH SỬ CHO LƯỢT TIẾP THEO
        chat_history.append({"role": "user", "content": current_turn_prompt})
        chat_history.append({"role": "assistant", "content": answer})
        
    except Exception as e:
        print(f"\n[Lỗi Ollama]: {e}")
        print("Mẹo: Đảm bảo ứng dụng Ollama đang chạy ngầm và bạn đã pull model thành công!")